# 📚 Motor de Recomendación de Libros con NLP

**Sistema de recomendaciones basado en TF-IDF + KNN con score compuesto ponderado**

Este notebook demuestra un motor de recomendación de libros que combina tres señales:
- **Contenido (45%)**: Similitud semántica entre descripciones usando TF-IDF con bigrams
- **Categoría (45%)**: Match de género/temática normalizado mediante diccionario de sinónimos
- **Autor (10%)**: Bonus cuando comparten escritor

**Técnicas utilizadas:**
- TF-IDF (Term Frequency–Inverse Document Frequency) con bigrams y sublinear TF
- NearestNeighbors (KNN) con distancia coseno
- Stopwords bilingües (español + inglés) + stopwords de dominio editorial
- Normalización de categorías WooCommerce mediante diccionario de sinónimos

---

> ⚠️ **Nota:** Este notebook trabaja con un dataset CSV anonimizado. Los SKUs están hasheados por seguridad.
> No se puede filtrar por stock ni disponibilidad en tienda — todos los libros son candidatos.

## 0. Instalación de Dependencias

Kaggle ya incluye `pandas`, `scikit-learn`, `numpy` y `tqdm`.  
No se necesitan instalaciones adicionales.

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")

## 1. Configuración del Modelo

Parámetros que controlan el comportamiento del motor de recomendaciones.

| Parámetro | Valor | Descripción |
|---|---|---|
| `PESO_CONTENIDO` | 0.45 | Peso del score TF-IDF (similitud semántica) |
| `PESO_CATEGORIA` | 0.45 | Peso del match de categoría (grupo canónico) |
| `PESO_AUTOR` | 0.10 | Peso del match de autor |
| `UMBRAL_SIMILITUD_MINIMA` | 0.01 | Score mínimo para aceptar recomendación |
| `CANTIDAD_RECOMENDACIONES` | 20 | Top N recomendaciones por libro |

In [ ]:
# =====================================================
# Configuración del modelo
# =====================================================

# Score mínimo para aceptar una recomendación (debajo de esto se descarta).
# Un valor muy bajo (0.01) permite cubrir libros con pocos candidatos elegibles.
UMBRAL_SIMILITUD_MINIMA = 0.01

# Pesos del score compuesto. Deben sumar 1.0.
# La categoría tiene peso alto (0.45) porque es la señal más confiable
# para evitar recomendaciones absurdas (ej. novela → cocina).
# El contenido (TF-IDF) captura similitud semántica entre descripciones.
# El autor da un bonus cuando el mismo escritor tiene múltiples obras.
PESO_CONTENIDO = 0.45
PESO_CATEGORIA = 0.45
PESO_AUTOR = 0.10

# Top N recomendaciones por libro
CANTIDAD_RECOMENDACIONES = 20

# Cantidad de vecinos cercanos a recuperar por libro desde NearestNeighbors.
# Se usa un margen amplio (×15) porque muchos candidatos se filtran después
# por idioma, elegibilidad y deduplicación de títulos.
CANDIDATOS_POR_LIBRO = CANTIDAD_RECOMENDACIONES * 15

print(f"⚙️ Configuración:")
print(f"   Peso contenido (TF-IDF): {PESO_CONTENIDO}")
print(f"   Peso categoría:          {PESO_CATEGORIA}")
print(f"   Peso autor:              {PESO_AUTOR}")
print(f"   Umbral mínimo:           {UMBRAL_SIMILITUD_MINIMA}")
print(f"   Recomendaciones/libro:    {CANTIDAD_RECOMENDACIONES}")
print(f"   Candidatos KNN/libro:     {CANDIDATOS_POR_LIBRO}")

## 2. Stopwords Bilingües

Se incluyen stopwords en **español e inglés** porque el TF-IDF se entrena sobre todo el catálogo (ambos idiomas juntos).  
Sin stopwords en inglés, palabras como *"the"*, *"and"*, *"book"* crearían ruido en la matriz de features.

Además de las stopwords gramaticales estándar, se incluyen **stopwords de dominio editorial** que aparecen en casi todas las descripciones de libros y no aportan señal discriminativa entre géneros (ej. *"libro"*, *"edición"*, *"páginas"*, *"book"*, *"story"*, *"novel"*).

In [ ]:
# =====================================================
# Stopwords bilingües (español + inglés)
# =====================================================
STOPWORDS_ESPANOL = [
    # Artículos, preposiciones, pronombres y conjunciones del español
    'el', 'la', 'de', 'que', 'y', 'a', 'en', 'un', 'una', 'por', 'con', 'para',
    'su', 'se', 'del', 'las', 'los', 'al', 'lo', 'como', 'más', 'pero', 'sus',
    'le', 'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy',
    'sin', 'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde',
    'todo', 'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros',
    'ese', 'eso', 'ante', 'ellos', 'esto', 'mí', 'antes', 'algunos', 'qué',
    'unos', 'yo', 'otro', 'otras', 'otra', 'él', 'tanto', 'esa', 'estos',
    'mucho', 'quienes', 'nada', 'muchos', 'cual', 'poco', 'ella', 'estar',
    'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tú', 'te', 'ti',
    'tu', 'tus', 'ellas', 'nosotras', 'es', 'soy', 'eres', 'somos', 'son',
    'fui', 'fue', 'fueron', 'ha', 'han', 'ser', 'sido', 'tiene', 'tienen',
    'había', 'haber', 'hacer', 'puede', 'siendo', 'así', 'después',
    # Palabras del dominio literario/editorial
    'historia', 'vida', 'mundo', 'años', 'días', 'hombres',
    'libro', 'libros', 'edición', 'editorial', 'tomo', 'colección', 'serie',
    'autor', 'leer', 'lectura', 'página', 'páginas', 'obra', 'obras',
    'texto', 'textos', 'capítulo', 'capítulos', 'volumen',
    'nuevo', 'nueva', 'nuevos', 'nuevas', 'gran', 'grande', 'grandes',
    'primer', 'primera', 'mejor', 'manera', 'forma', 'parte', 'tiempo',
    'lugar', 'cada', 'través', 'cuenta', 'hombre', 'mujer',
    'dos', 'tres', 'cuatro', 'cinco', 'seis', 'siete', 'ocho', 'nueve', 'diez',
]

STOPWORDS_INGLES = [
    # Equivalentes en inglés: artículos, preposiciones, pronombres
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'it', 'as', 'was', 'are', 'be',
    'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
    'would', 'could', 'should', 'may', 'might', 'must', 'shall', 'can',
    'not', 'no', 'nor', 'so', 'if', 'then', 'than', 'too', 'very',
    'just', 'about', 'above', 'after', 'again', 'all', 'also', 'am',
    'any', 'because', 'before', 'between', 'both', 'each', 'few',
    'further', 'get', 'got', 'he', 'her', 'here', 'him', 'his', 'how',
    'into', 'its', 'let', 'me', 'more', 'most', 'my', 'now', 'off',
    'only', 'other', 'our', 'out', 'over', 'own', 'same', 'she', 'some',
    'still', 'such', 'that', 'their', 'them', 'there', 'these', 'they',
    'this', 'those', 'through', 'under', 'until', 'up', 'us', 'we',
    'what', 'when', 'where', 'which', 'while', 'who', 'whom', 'why',
    'you', 'your', 'yours', 'yourself',
    # Palabras del dominio editorial en inglés
    'book', 'books', 'story', 'stories', 'novel', 'edition', 'chapter',
    'author', 'read', 'reading', 'page', 'pages', 'volume', 'series',
    'new', 'great', 'first', 'one', 'two', 'three', 'way', 'time',
    'world', 'life', 'year', 'years', 'man', 'woman', 'day', 'days',
    'many', 'much', 'well', 'back', 'even', 'also', 'made', 'make',
    'like', 'set', 'part', 'long', 'find', 'work', 'come', 'take',
]

# Se combinan ambas listas para alimentar el TfidfVectorizer
STOPWORDS_BILINGUE = STOPWORDS_ESPANOL + STOPWORDS_INGLES

print(f"📝 Stopwords cargadas:")
print(f"   Español: {len(STOPWORDS_ESPANOL)} palabras")
print(f"   Inglés:  {len(STOPWORDS_INGLES)} palabras")
print(f"   Total:   {len(STOPWORDS_BILINGUE)} stopwords bilingües")

## 3. Normalización de Categorías

Las categorías en el catálogo no están normalizadas: existen variantes en español e inglés, con distintas mayúsculas y separadores.

**Ejemplo:**
- `"Children's Fiction"` (1001 libros)
- `"Libros Infantiles"` (607)
- `"Children Fiction"` (190)
- `"Children Book"` (120)

→ Todos representan el mismo género: **`infantil`**

El diccionario de sinónimos mapea cada variante conocida a un **grupo canónico** (18 grupos definidos).

In [ ]:
# =====================================================
# Mapeo de sinónimos de categorías
# =====================================================
CATEGORIA_SINONIMOS = {
    'infantil': [
        "Children's Fiction", "Children Fiction", "Children Book",
        "Children's Non-Fiction", "Children Non-Fiction",
        "Libros Infantiles", "Libros Infantiles en Español",
        "Cuentos para niños", "Cuentos Infantiles", "Board books",
        "Kids & Children", "Kids (12 & Under)",
        "Libros para bebés", "Libros de actividades", "Activity Books",
        "I CAN READ",
    ],
    'juvenil': [
        "Juvenile Fiction", "Juvenile Nonfiction",
        "Novelas Juveniles", "Young Adult Fiction", "Young Adult",
        "Tweens Fiction", "Kids: Middle Grade", "Kids Middle Grade",
    ],
    'ficcion': [
        "Fiction", "General Fiction", "Novels", "Novelas",
    ],
    'accion_aventura': [
        "Action & Adventure", "Action", "Adventure",
    ],
    'fantasia_scifi': [
        "Fantasy", "Science Fiction",
    ],
    'cocina': [
        "Cook Book", "Cocina",
    ],
    'religion': [
        "Religion", "Religión y Espiritualidad", "Religion & Spirituality",
    ],
    'comics': [
        "Cómics de Colección", "Comics & Graphic Novels",
    ],
    'arte_fotografia': [
        "Arte", "Fotografía y diseño", "Art & Photography",
    ],
    'historia_cultura': [
        "Historia y Cultura",
    ],
    'cuentos': [
        "Cuentos",
    ],
    'aprendizaje': [
        "Aprendizaje", "Learning Books",
    ],
    'animales': [
        "Libros de animales", "Animals",
    ],
    'bienestar_salud': [
        "Bienestar y Salud",
    ],
    'colorear': [
        "Libros para colorear",
    ],
    'navidad': [
        "Libros de Navidad", "Christmas Books",
    ],
    'halloween': [
        "Libros de Halloween para Niños",
    ],
    'conejos': [
        "Libros de Conejos",
    ],
}

# Categorías que son ruido: no describen temática/género sino promociones,
# temporadas o rangos de precio.
CATEGORIAS_RUIDO = [
    "Novedades", "2025", "Easter",
    "Día de la Madre", "Día de la mujer",
    "Libros hasta 9.90", "Libros en Español",
]

total_variantes = sum(len(v) for v in CATEGORIA_SINONIMOS.values())
print(f"📂 Categorías configuradas:")
print(f"   {len(CATEGORIA_SINONIMOS)} grupos canónicos")
print(f"   {total_variantes} variantes mapeadas")
print(f"   {len(CATEGORIAS_RUIDO)} categorías de ruido filtradas")

## 4. Funciones Utilitarias

In [ ]:
# =====================================================
# Funciones utilitarias
# =====================================================

# Regex para separar strings con múltiples autores (ej. "García, Borges | Paz")
SEPARADORES_AUTORES = r'[,|;/&]'


def limpiar_html(texto):
    """Elimina etiquetas HTML de las descripciones (ej. <p>, <br>, <strong>)."""
    return re.sub(r'<[^>]+>', ' ', str(texto))


def extraer_titulo_base(titulo):
    """Extrae el título base de un libro para detectar duplicados (distintas ediciones)."""
    t = str(titulo).lower().strip()
    t = re.split(r'[:\-—]|\bpor\b|\bby\b', t)[0]
    t = re.sub(r'^(the|a|an|el|la|los|las|un|una)\s+', '', t.strip())
    t = re.sub(r',\s*(the|a|an|el|la|los|las|un|una)$', '', t.strip())
    t = re.sub(r'[^\w\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def normalizar_categoria(cat):
    """Normaliza una categoría para comparación flexible."""
    cat = str(cat).strip().lower()
    cat = re.sub(r'[>\-/|,.:;()\[\]{}\'\"\ &]', ' ', cat)
    cat = re.sub(r'\s+', ' ', cat).strip()
    return cat


# Pre-construir diccionario invertido:
# {"children s fiction": "infantil", "libros infantiles": "infantil", ...}
# Esto permite búsquedas O(1) en vez de iterar el diccionario original.
_MAPA_CAT_A_GRUPO = {}
for _grupo, _variantes in CATEGORIA_SINONIMOS.items():
    for _variante in _variantes:
        _MAPA_CAT_A_GRUPO[normalizar_categoria(_variante)] = _grupo

_CATEGORIAS_RUIDO_NORM = {normalizar_categoria(c) for c in CATEGORIAS_RUIDO}


def get_grupo_categoria(cat_normalizada):
    """Mapea una categoría normalizada a su grupo canónico.
    Retorna None si es una categoría de ruido (sin señal temática).
    Retorna la categoría misma como fallback si no está en el mapeo."""
    if not cat_normalizada or cat_normalizada in _CATEGORIAS_RUIDO_NORM:
        return None
    if cat_normalizada in _MAPA_CAT_A_GRUPO:
        return _MAPA_CAT_A_GRUPO[cat_normalizada]
    for variante_norm, grupo in _MAPA_CAT_A_GRUPO.items():
        if variante_norm in cat_normalizada:
            return grupo
    return cat_normalizada


def score_categoria(grupo_a, grupo_b):
    """1.0 si ambos pertenecen al mismo grupo, 0.0 si no."""
    if not grupo_a or not grupo_b:
        return 0.0
    return 1.0 if grupo_a == grupo_b else 0.0


def extraer_set_autores(autor_raw):
    """Separa un string de múltiples autores en un set normalizado."""
    if not autor_raw or str(autor_raw).strip() == '':
        return set()
    partes = re.split(SEPARADORES_AUTORES, str(autor_raw))
    return {a.strip().lower() for a in partes if a.strip()}


def score_autor(autores_a, autores_b):
    """1.0 si comparten al menos un autor en común, 0.0 si no."""
    if not autores_a or not autores_b:
        return 0.0
    return 1.0 if autores_a & autores_b else 0.0


print("✅ Funciones utilitarias cargadas")
print(f"   {len(_MAPA_CAT_A_GRUPO)} variantes de categoría mapeadas")
print(f"   {len(_CATEGORIAS_RUIDO_NORM)} categorías de ruido")

## 5. Carga del Dataset

En lugar de conectarse a una base de datos MySQL, este notebook lee un CSV anonimizado exportado desde el sistema de producción.

**Columnas del CSV:**
| Columna | Descripción |
|---|---|
| `sku_anonimo` | SKU hasheado (SHA-256, primeros 16 caracteres) |
| `title` | Título del libro |
| `author` | Autor(es) del libro |
| `grupo_categoria` | Grupo canónico de categoría (ya normalizado) |
| `language` | Idioma del libro |
| `Texto_IA_Limpio` | Descripción limpia (sin HTML) |

In [ ]:
# =====================================================
# 1. Carga del dataset CSV
# =====================================================
# En Kaggle, subí el archivo como dataset → se monta en /kaggle/input/
# ⚠️ Ajustá esta ruta según el nombre de tu dataset en Kaggle:
#    /kaggle/input/{nombre-del-dataset}/{nombre-del-archivo}.csv

# Ruta para Kaggle (ajustar nombre del dataset)
RUTA_CSV = '/kaggle/input/book-catalog-features/book_catalog_features_kaggle.csv'

# Ruta alternativa para ejecución local
# RUTA_CSV = 'book_catalog_features_kaggle.csv'

print("1. Cargando dataset desde CSV...")
try:
    df = pd.read_csv(RUTA_CSV)
    print(f"   ✅ Dataset cargado: {len(df)} libros")
except FileNotFoundError:
    print(f"   ❌ Archivo no encontrado: {RUTA_CSV}")
    print(f"   Asegurate de subir 'book_catalog_features_kaggle.csv' como dataset en Kaggle")
    print(f"   o ajustá la variable RUTA_CSV al path correcto.")
    raise

print(f"\n📋 Columnas: {list(df.columns)}")
print(f"\n📊 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head(3)

## 6. Análisis Exploratorio del Dataset

In [ ]:
# =====================================================
# Análisis exploratorio
# =====================================================
print("📊 Estadísticas del Dataset")
print("=" * 50)

# Distribución de idiomas
print("\n🌐 Distribución por Idioma:")
idiomas = df['language'].value_counts()
for idioma, count in idiomas.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"   {idioma:15s} {count:5d} ({pct:5.1f}%) {bar}")

# Distribución de categorías
print(f"\n📂 Top 10 Categorías (grupos canónicos):")
categorias = df['grupo_categoria'].value_counts().head(10)
for cat, count in categorias.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"   {str(cat):20s} {count:5d} ({pct:5.1f}%) {bar}")

# Valores nulos
print(f"\n⚠️ Valores nulos:")
nulos = df.isnull().sum()
for col, count in nulos.items():
    status = '✅' if count == 0 else '⚠️'
    print(f"   {status} {col}: {count}")

# Longitud de descripciones
desc_lengths = df['Texto_IA_Limpio'].fillna('').str.len()
print(f"\n📝 Longitud de descripciones:")
print(f"   Promedio: {desc_lengths.mean():.0f} caracteres")
print(f"   Mediana:  {desc_lengths.median():.0f} caracteres")
print(f"   Mínima:   {desc_lengths.min():.0f} caracteres")
print(f"   Máxima:   {desc_lengths.max():.0f} caracteres")

## 7. Procesamiento de Textos

Se prepara el texto de entrada para el modelo TF-IDF:
1. Relleno de valores nulos
2. Limpieza de HTML residual
3. Construcción del texto combinado: `author + title + descripción`
4. Precálculo de grupos de categoría y sets de autores

In [ ]:
# =====================================================
# 2. Procesamiento de textos
# =====================================================
print("2. Procesando textos y normalizando...")

# Rellenar valores nulos para evitar errores en concatenación y comparación
df['title'] = df['title'].fillna('')
df['author'] = df['author'].fillna('')
df['grupo_categoria'] = df['grupo_categoria'].fillna('')
df['Texto_IA_Limpio'] = df['Texto_IA_Limpio'].fillna('')
df['language'] = df['language'].fillna('Desconocido')

# Limpiar HTML residual que pudiera quedar en las descripciones
df['Texto_IA_Limpio'] = df['Texto_IA_Limpio'].apply(limpiar_html)

# Construir el texto de entrada para TF-IDF.
# IMPORTANTE: La categoría NO se incluye aquí.
# Se evalúa como score separado para evitar contaminar el TF-IDF
# con tokens de categoría que distorsionan las estadísticas de frecuencia.
df['Texto_IA'] = df['author'] + " " + df['title'] + " " + df['Texto_IA_Limpio']

# El CSV ya tiene 'grupo_categoria' pre-calculado.
# Sin embargo, valores vacíos se tratan como None para el scoring.
df['grupo_categoria'] = df['grupo_categoria'].replace('', None)

# Precalcular sets de autores separando por delimitadores
df['autores_set'] = df['author'].apply(extraer_set_autores)

# Convertir columnas a arrays NumPy para acceso rápido en el loop principal
lista_ids = df['sku_anonimo'].astype(str).values
lista_grupos_cat = df['grupo_categoria'].values
lista_autores_set = df['autores_set'].values
lista_idiomas = df['language'].astype(str).values
lista_textos_limpios = df['Texto_IA_Limpio'].str.strip().values
titulos_base = [extraer_titulo_base(t) for t in df['title'].values]
lista_titulos = df['title'].values

# Diagnóstico
grupos_encontrados = df['grupo_categoria'].dropna().nunique()
sin_grupo = df['grupo_categoria'].isna().sum()
print(f"   → {grupos_encontrados} grupos de categoría detectados")
print(f"   → {sin_grupo} libros con categoría de ruido/sin categoría")
print(f"   → {len(df)} textos listos para TF-IDF")

## 8. Entrenamiento del Modelo TF-IDF

> **TF-IDF (Term Frequency–Inverse Document Frequency)** convierte cada texto en un vector numérico donde cada dimensión representa la importancia de un término (o bigrama) para ese documento relativo al resto del corpus.

**Hiperparámetros:**
- `max_df=0.80`: Ignora términos en >80% de documentos (demasiado comunes)
- `min_df=2`: Ignora términos en <2 documentos (demasiado raros)
- `ngram_range=(1, 2)`: Captura unigramas y bigramas (ej. *"guerra civil"*, *"amor prohibido"*)
- `sublinear_tf=True`: Aplica `log(1 + tf)`, reduce impacto de repetición
- `max_features=50000`: Limita vocabulario para controlar memoria
- `token_pattern`: Exige al menos una letra por token, filtrando tokens puramente numéricos

In [ ]:
# =====================================================
# 3. Entrenar modelo TF-IDF (bigrams + sublinear_tf)
# =====================================================
print("3. Entrenando el modelo TF-IDF (bigrams + sublinear_tf + stopwords bilingues)...")

vectorizer = TfidfVectorizer(
    max_df=0.80,         # Ignora términos en >80% de documentos
    min_df=2,            # Ignora términos en <2 documentos
    stop_words=STOPWORDS_BILINGUE,
    ngram_range=(1, 2),  # Unigramas y bigramas
    sublinear_tf=True,   # log(1 + tf)
    max_features=50000,  # Limita vocabulario
    # Exige que cada token EMPIECE con una letra, descartando tokens numéricos
    # o con prefijo numérico ('000', '100th', '1000stickers') que vienen de
    # marketing y no aportan señal discriminativa entre géneros.
    token_pattern=r'(?u)\b[a-zA-ZáéíóúñüÀ-ÿ]\w+\b'

)

tfidf_matrix = vectorizer.fit_transform(df['Texto_IA'])

print(f"   ✅ Matriz TF-IDF generada")
print(f"   Dimensiones: {tfidf_matrix.shape[0]} documentos × {tfidf_matrix.shape[1]} features")
print(f"   Densidad: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100:.2f}%")
print(f"   Memoria (aprox): {tfidf_matrix.data.nbytes / 1024 / 1024:.1f} MB (datos sparse)")

# Mostrar algunos features del vocabulario
vocab = vectorizer.get_feature_names_out()
print(f"\n📖 Vocabulario: {len(vocab)} términos")
print(f"   Primeros 10: {list(vocab[:10])}")
print(f"   Últimos 10:  {list(vocab[-10:])}")

## 9. Entrenamiento del Modelo NearestNeighbors

> En vez de calcular la similitud coseno de CADA libro contra TODOS los demás (**O(n²)**, crece cuadráticamente con el catálogo), **NearestNeighbors** pre-indexa los vectores y busca solo los K vecinos más cercanos de forma optimizada.

In [ ]:
# =====================================================
# 4. Entrenar modelo NearestNeighbors
# =====================================================
print("4. Entrenando modelo de vecinos cercanos (NearestNeighbors)...")

n_neighbors = min(CANDIDATOS_POR_LIBRO, len(df) - 1)
print(f"   K vecinos a buscar: {n_neighbors}")

nn_model = NearestNeighbors(
    n_neighbors=n_neighbors,
    metric='cosine',     # Distancia coseno: 0 = idénticos, 2 = opuestos
    algorithm='brute',   # Búsqueda exacta (más precisa que approximate NN)
    n_jobs=-1            # Usa todos los cores del CPU en paralelo
)
nn_model.fit(tfidf_matrix)

# Obtener todos los vecinos de una sola vez (batch)
print("   Calculando vecinos para todo el catálogo (batch)...")
distances, indices = nn_model.kneighbors(tfidf_matrix)

# Convertir distancia coseno → similitud coseno (distancia = 1 - similitud)
similarities = 1 - distances

print(f"   ✅ Vecinos calculados")
print(f"   Forma distances: {distances.shape}")
print(f"   Rango similitud: [{similarities.min():.4f}, {similarities.max():.4f}]")

## 10. Generación de Recomendaciones

Para cada libro, se recorren sus K vecinos más cercanos y se calcula un **score compuesto**:

$$\text{score} = 0.45 \times \text{contenido}_{\text{TF-IDF}} + 0.45 \times \text{categoría} + 0.10 \times \text{autor}$$

**Filtros aplicados a los candidatos:**
1. No recomendarse a sí mismo
2. Descartar libros sin descripción útil
3. Evitar recomendar la misma obra en otra edición (deduplicación por título base)
4. Solo recomendar libros en el mismo idioma
5. Score mínimo ≥ 0.01

> ⚠️ **Nota Kaggle:** En producción también se filtra por stock > 0 y disponibilidad en WooCommerce, pero esos datos NO están en el CSV.

In [ ]:
# =====================================================
# 5. Generar recomendaciones con score compuesto
# =====================================================
print("5. Generando recomendaciones (score compuesto)...")

# Almacenar todas las recomendaciones en una lista de diccionarios
todas_recomendaciones = []
libros_sin_recs = 0
total_recs = 0

for idx in tqdm(range(len(df)), desc="Procesando Catálogo"):
    sku_actual = lista_ids[idx]
    grupo_cat_actual = lista_grupos_cat[idx]
    autores_actual = lista_autores_set[idx]
    idioma_actual = lista_idiomas[idx]
    titulo_base_actual = titulos_base[idx]

    candidatos = []

    # Recorrer los K vecinos más cercanos pre-calculados por NearestNeighbors
    for j_pos in range(len(indices[idx])):
        i = indices[idx][j_pos]
        sim_contenido = float(similarities[idx][j_pos])

        # --- Filtros de exclusión ---
        if i == idx:                                    # No recomendarse a sí mismo
            continue
        if not lista_textos_limpios[i]:                  # Sin descripción útil
            continue
        if titulos_base[i] == titulo_base_actual:        # Misma obra, distinta edición
            continue
        if lista_idiomas[i] != idioma_actual:            # Distinto idioma
            continue
        # NOTA: No se filtra por stock/WC — datos no disponibles en CSV

        # --- Cálculo del score compuesto ---
        s_cat = score_categoria(grupo_cat_actual, lista_grupos_cat[i])
        s_autor = score_autor(autores_actual, lista_autores_set[i])

        score_final = (
            PESO_CONTENIDO * sim_contenido
            + PESO_CATEGORIA * s_cat
            + PESO_AUTOR * s_autor
        )

        if score_final < UMBRAL_SIMILITUD_MINIMA:
            continue

        candidatos.append((lista_ids[i], score_final, i))

    # Ordenar por score compuesto descendente
    candidatos.sort(key=lambda x: x[1], reverse=True)
    recomendados = candidatos[:CANTIDAD_RECOMENDACIONES]

    if len(recomendados) == 0:
        libros_sin_recs += 1

    total_recs += len(recomendados)

    for rank, (rec_sku, score, rec_idx) in enumerate(recomendados, start=1):
        todas_recomendaciones.append({
            'source_sku': sku_actual,
            'source_title': lista_titulos[idx],
            'recommended_sku': rec_sku,
            'recommended_title': lista_titulos[rec_idx],
            'rank_order': rank,
            'similarity_score': round(score, 4)
        })

# Crear DataFrame con los resultados
df_recomendaciones = pd.DataFrame(todas_recomendaciones)

promedio_recs = total_recs / len(df) if len(df) > 0 else 0
print(f"\n✅ Proceso finalizado")
print(f"📊 Estadísticas:")
print(f"   {total_recs:,} recomendaciones generadas")
print(f"   Promedio: {promedio_recs:.1f} recomendaciones por libro")
if libros_sin_recs > 0:
    print(f"   ⚠️ {libros_sin_recs} libros sin recomendaciones (sin candidatos elegibles)")

## 11. Exploración de Resultados

In [ ]:
# =====================================================
# Vista general de las recomendaciones
# =====================================================
print("📋 Tabla de Recomendaciones (primeras 20 filas):")
df_recomendaciones.head(20)

In [ ]:
# =====================================================
# Distribución de scores
# =====================================================
print("📊 Distribución de Scores de Similitud:")
print(f"   Media:    {df_recomendaciones['similarity_score'].mean():.4f}")
print(f"   Mediana:  {df_recomendaciones['similarity_score'].median():.4f}")
print(f"   Std:      {df_recomendaciones['similarity_score'].std():.4f}")
print(f"   Mín:      {df_recomendaciones['similarity_score'].min():.4f}")
print(f"   Máx:      {df_recomendaciones['similarity_score'].max():.4f}")

# Histograma de distribución de scores (texto, sin matplotlib)
print("\n📈 Histograma de Scores:")
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
hist = pd.cut(df_recomendaciones['similarity_score'], bins=bins).value_counts().sort_index()
max_count = hist.max()
for rango, count in hist.items():
    bar_len = int(count / max_count * 40) if max_count > 0 else 0
    bar = '█' * bar_len
    print(f"   {str(rango):15s} {count:6,} {bar}")

In [ ]:
# =====================================================
# Cantidad de recomendaciones por libro
# =====================================================
recs_por_libro = df_recomendaciones.groupby('source_sku').size()
print("📊 Recomendaciones por Libro:")
print(f"   Libros con recomendaciones: {len(recs_por_libro):,}")
print(f"   Libros sin recomendaciones: {libros_sin_recs:,}")
print(f"   Promedio recs/libro:        {recs_por_libro.mean():.1f}")
print(f"   Mediana recs/libro:         {recs_por_libro.median():.1f}")
print(f"   Mín recs/libro:             {recs_por_libro.min()}")
print(f"   Máx recs/libro:             {recs_por_libro.max()}")
print(f"\n📊 Distribución:")
dist = recs_por_libro.value_counts().sort_index()
for n_recs, count in dist.items():
    bar = '█' * (count // max(1, dist.max() // 40))
    print(f"   {n_recs:2d} recs → {count:5,} libros {bar}")

## 12. Demo Interactiva: Buscar Recomendaciones para un Libro

In [ ]:
def mostrar_recomendaciones(titulo_busqueda, top_n=10):
    """
    Busca un libro por título (búsqueda parcial, case-insensitive)
    y muestra sus recomendaciones.
    """
    # Buscar libros que coincidan con la búsqueda
    mask = df['title'].str.contains(titulo_busqueda, case=False, na=False)
    matches = df[mask]
    
    if len(matches) == 0:
        print(f"❌ No se encontró ningún libro con '{titulo_busqueda}' en el título")
        return
    
    if len(matches) > 1:
        print(f"📚 Se encontraron {len(matches)} coincidencias:")
        for _, row in matches.head(5).iterrows():
            print(f"   • {row['title']} ({row['language']}) — SKU: {row['sku_anonimo'][:8]}...")
        print(f"   Usando el primero...\n")
    
    # Tomar el primer match
    libro = matches.iloc[0]
    sku = libro['sku_anonimo']
    
    print("=" * 70)
    print(f"📖 {libro['title']}")
    print(f"   Autor:     {libro['author']}")
    print(f"   Categoría: {libro['grupo_categoria']}")
    print(f"   Idioma:    {libro['language']}")
    print("=" * 70)
    
    # Buscar recomendaciones
    recs = df_recomendaciones[df_recomendaciones['source_sku'] == sku].head(top_n)
    
    if len(recs) == 0:
        print(f"⚠️ No hay recomendaciones para este libro")
        return
    
    print(f"\n🎯 Top {min(top_n, len(recs))} Recomendaciones:\n")
    
    for _, rec in recs.iterrows():
        # Buscar info del libro recomendado en el dataframe original
        rec_info = df[df['sku_anonimo'] == rec['recommended_sku']]
        if len(rec_info) > 0:
            rec_libro = rec_info.iloc[0]
            score_bar = '█' * int(rec['similarity_score'] * 20)
            print(f"   {rec['rank_order']:2d}. {rec['recommended_title']}")
            print(f"       Autor: {rec_libro['author']}")
            print(f"       Cat:   {rec_libro['grupo_categoria']} | Idioma: {rec_libro['language']}")
            print(f"       Score: {rec['similarity_score']:.4f} {score_bar}")
            print()

In [ ]:
# =====================================================
# Ejemplo: Buscar recomendaciones para un libro
# =====================================================
# Probá con distintos títulos del catálogo:

# Mostremos primeros 5 títulos del catálogo como ejemplo
print("📚 Títulos de ejemplo del catálogo:")
for i, titulo in enumerate(df['title'].dropna().unique()[:10]):
    print(f"   {i+1}. {titulo}")

print("\n" + "─" * 70)
print("Buscando recomendaciones para el primer libro del catálogo...\n")

# Usar el primer título como demostración
primer_titulo = df['title'].dropna().iloc[0]
mostrar_recomendaciones(primer_titulo, top_n=10)

In [ ]:
# =====================================================
# Probar con otro libro (cambiá el título acá)
# =====================================================
# Escribí parte del título de cualquier libro del catálogo:

# mostrar_recomendaciones("Harry Potter", top_n=5)
# mostrar_recomendaciones("cocina", top_n=5)
# mostrar_recomendaciones("dragon", top_n=5)

## 13. Análisis de Calidad del Modelo

In [ ]:
# =====================================================
# Análisis de calidad: ¿Las recomendaciones son coherentes?
# =====================================================
print("🔍 Análisis de Calidad del Modelo")
print("=" * 50)

# 1. ¿Cuántas recomendaciones comparten categoría con el libro origen?
def analizar_coherencia_categorias():
    total = 0
    misma_cat = 0
    
    for _, rec in df_recomendaciones.iterrows():
        source = df[df['sku_anonimo'] == rec['source_sku']]
        target = df[df['sku_anonimo'] == rec['recommended_sku']]
        
        if len(source) > 0 and len(target) > 0:
            total += 1
            cat_source = source.iloc[0]['grupo_categoria']
            cat_target = target.iloc[0]['grupo_categoria']
            if cat_source and cat_target and cat_source == cat_target:
                misma_cat += 1
    
    pct = (misma_cat / total * 100) if total > 0 else 0
    print(f"\n📂 Coherencia de Categorías:")
    print(f"   {misma_cat:,} / {total:,} recomendaciones comparten categoría ({pct:.1f}%)")
    return pct

# 2. ¿Cuántas recomendaciones comparten autor?
def analizar_coherencia_autores():
    total = 0
    mismo_autor = 0
    
    for _, rec in df_recomendaciones.iterrows():
        source = df[df['sku_anonimo'] == rec['source_sku']]
        target = df[df['sku_anonimo'] == rec['recommended_sku']]
        
        if len(source) > 0 and len(target) > 0:
            total += 1
            autores_s = extraer_set_autores(source.iloc[0]['author'])
            autores_t = extraer_set_autores(target.iloc[0]['author'])
            if autores_s & autores_t:
                mismo_autor += 1
    
    pct = (mismo_autor / total * 100) if total > 0 else 0
    print(f"\n✍️ Coherencia de Autores:")
    print(f"   {mismo_autor:,} / {total:,} recomendaciones comparten autor ({pct:.1f}%)")
    return pct

# 3. Score promedio por rango
print("\n📊 Score Promedio por Posición en el Ranking:")
score_por_rank = df_recomendaciones.groupby('rank_order')['similarity_score'].mean()
for rank, avg_score in score_por_rank.head(20).items():
    bar = '█' * int(avg_score * 40)
    print(f"   Rank {rank:2d}: {avg_score:.4f} {bar}")

# Ejecutar análisis (puede tomar un momento en catálogos grandes)
# ⚠️ Si el catálogo es muy grande (>5000 libros), estos análisis toman tiempo.
# Descomenta las siguientes líneas para ejecutarlos:

# pct_cat = analizar_coherencia_categorias()
# pct_aut = analizar_coherencia_autores()

print("\n💡 Tip: Descomenta las funciones analizar_coherencia_* para un análisis completo")
print("   (puede tomar unos minutos en catálogos grandes)")

## 14. Visualización de la Matriz TF-IDF (Top Términos)

In [ ]:
# =====================================================
# Top términos más importantes por categoría
# =====================================================
print("📊 Top 10 Términos TF-IDF más Importantes por Categoría")
print("=" * 60)

feature_names = vectorizer.get_feature_names_out()
categorias_top = df['grupo_categoria'].dropna().value_counts().head(5).index

for cat in categorias_top:
    # Seleccionar libros de esta categoría
    mask_cat = df['grupo_categoria'] == cat
    indices_cat = df[mask_cat].index.tolist()
    
    if len(indices_cat) == 0:
        continue
    
    # Promediar los vectores TF-IDF de la categoría
    tfidf_cat = tfidf_matrix[indices_cat].mean(axis=0)
    tfidf_cat = np.asarray(tfidf_cat).flatten()
    
    # Top 10 términos
    top_indices = tfidf_cat.argsort()[-10:][::-1]
    
    print(f"\n📂 {cat} ({len(indices_cat)} libros):")
    for i, idx_term in enumerate(top_indices):
        score = tfidf_cat[idx_term]
        bar = '█' * int(score * 100)
        print(f"   {i+1:2d}. {feature_names[idx_term]:25s} {score:.4f} {bar}")

## 15. Exportación de Resultados

In [ ]:
# =====================================================
# Exportar recomendaciones a CSV
# =====================================================
output_file = 'book_recommendations_output.csv'
df_recomendaciones.to_csv(output_file, index=False)

print(f"💾 Recomendaciones exportadas a: {output_file}")
print(f"   {len(df_recomendaciones):,} filas")
print(f"   Columnas: {list(df_recomendaciones.columns)}")

# Resumen final
print("\n" + "=" * 60)
print("📊 RESUMEN FINAL")
print("=" * 60)
print(f"   Libros en catálogo:         {len(df):,}")
print(f"   Recomendaciones generadas:  {len(df_recomendaciones):,}")
print(f"   Promedio recs/libro:        {promedio_recs:.1f}")
print(f"   Libros sin recomendación:   {libros_sin_recs:,}")
print(f"   Features TF-IDF:            {tfidf_matrix.shape[1]:,}")
print(f"   Vecinos KNN por libro:      {n_neighbors}")
print(f"   Score promedio:             {df_recomendaciones['similarity_score'].mean():.4f}")
print(f"   Score mediano:              {df_recomendaciones['similarity_score'].median():.4f}")

---

## 📝 Notas Técnicas

### Diferencias con el Script de Producción

| Aspecto | Producción (`book_recommender.py`) | Kaggle (este notebook) |
|---|---|---|
| **Fuente de datos** | MySQL (SQLAlchemy) | CSV (pandas) |
| **SKUs** | Reales | Anonimizados (SHA-256) |
| **Filtro elegibilidad** | stock > 0 AND wc_product_id IS NOT NULL | No disponible |
| **Salida** | INSERT en tabla `product_recommendations` | DataFrame + CSV |
| **Categorías** | Se normalizan desde la columna `category` original | Ya vienen como `grupo_categoria` |
| **Dependencias** | SQLAlchemy, mysql-connector-python | Ninguna adicional |

### Arquitectura del Score Compuesto

```
                    ┌─────────────────┐
                    │  TF-IDF Matrix  │
                    │  (bigrams +     │
                    │   sublinear_tf) │
                    └────────┬────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │  KNN Neighbors  │
                    │  (cosine dist)  │
                    └────────┬────────┘
                             │
              ┌──────────────┼──────────────┐
              │              │              │
              ▼              ▼              ▼
     ┌──────────────┐ ┌───────────┐ ┌───────────┐
     │  Contenido   │ │ Categoría │ │   Autor   │
     │  (45%)       │ │ (45%)     │ │  (10%)    │
     │  0.0 → 1.0   │ │ 0 ó 1    │ │ 0 ó 1    │
     └──────┬───────┘ └─────┬─────┘ └─────┬─────┘
            │               │             │
            └───────────────┼─────────────┘
                            │
                            ▼
                   ┌─────────────────┐
                   │  Score Compuesto │
                   │  (0.0 → 1.0)   │
                   └─────────────────┘
```

### Sobre los Datos

- Los SKUs están anonimizados con SHA-256 + salt para proteger información comercial
- Los títulos y autores son públicos (aparecen en WooCommerce)
- Las descripciones son generadas/enriquecidas por otro módulo del sistema (`book_data_enrichment.py`)